# Reproduce the Bastion Prompt Protection benchmark

Run the full **detection leaderboard** and **false-positive-rate** suite on a **free Colab T4** — roughly 15–30 minutes. Every number in the README and on the model card is produced by these two scripts, on public models and public benchmarks; this notebook lets you verify them yourself.

**Before you start:** set a GPU runtime — *Runtime → Change runtime type → T4 GPU*.

## 1. Get the code + install

In [ ]:
!git clone --quiet https://github.com/bastion-soft/bastion-prompt-protection.git
%cd bastion-prompt-protection
!pip install -q -e ".[eval]"

## 2. (Optional) Hugging Face login

The token is **optional**. A few baselines/datasets are **gated** and need a free HF token with access granted:
- `meta-llama/Prompt-Guard-86M` (a baseline) — accept on its model page
- `lmsys/lmsys-chat-1m` (an FPR dataset) — accept its license
- `hackaprompt/hackaprompt-dataset` (an indirect set) — accept on its dataset page

Without a token these are **excluded — skipped cleanly** and the rest of the run continues unaffected. The free 70M model and every open baseline/dataset need no token.

In [ ]:
# Optional: add HF_TOKEN to Colab Secrets (key icon, left), or skip this cell.
from huggingface_hub import login

try:
    from google.colab import userdata

    login(userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN.")
except Exception:
    print("No HF_TOKEN found - gated models/datasets will be skipped (that's fine).")

## 3. Detection leaderboard

AUC + F1 across four held-out adversarial benchmarks (rogue, xTRam1, S-Labs, JailbreakBench). The script prints a markdown table and writes `eval/results/leaderboard.{json,md}`.

Tip: add `--limit 200` for a fast smoke run first.

In [ ]:
!python -m scripts.run_leaderboard --dump-scores eval/results/scores

## 4. False-positive rate

The half most comparisons skip: the share of **benign** real-user messages each model wrongly flags, on WildChat + LMSYS openers. Lower is better. Writes `eval/results/false_positives.json`.

Tip: add `--n 500` for a fast smoke run first.

In [ ]:
!python -m scripts.measure_false_positives --dump-scores eval/results/scores

## 5. Indirect / structured injection

Where most detectors fall off: injection hidden inside **data** — JSON/XML agent interactions (Z-Edgar), document context (BIPIA), poisoned tool outputs (InjecAgent), and agentic attacks (AgentDojo / HackAPrompt / TensorTrust). Scored pure-model, same as §3, on a **separate** table (a distinct capability axis — not folded into the §3 average). Writes `eval/results/indirect.{json,md}`.

HackAPrompt is gated (needs the HF token from §2); AgentDojo needs `pip install agentdojo` — both skip cleanly if unavailable.

In [ ]:
# AgentDojo ships as its own package (one of the indirect sets); install it here.
!pip install -q agentdojo
!python -m scripts.eval_indirect

## 6. Operating points (threshold-agnostic)

The fairest comparison: instead of a fixed 0.5 line, set each detector's threshold to the **same detection rate** (e.g. catch 95% of attacks), then report how much **real benign traffic** it flags at that catch rate — plus equal-error rate and a threshold sweep (0.2 / 0.45 / 0.5 / 0.55 / 0.8). Pure post-processing over the per-prompt scores dumped in §3 + §4 (no GPU, fully reproducible). Writes `eval/results/operating_points.{json,md}` and `det_points.json`.

In [ ]:
# Threshold-agnostic analysis over the per-prompt scores dumped in §3 + §4 (no GPU).
!python -m scripts.analyze_operating_points
# Optional curves (DET/operating + FPR-vs-threshold):
!pip install -q matplotlib && python -m scripts.plot_operating_points

## Results

- `eval/results/leaderboard.json` + `leaderboard.md` — detection (AUC / F1 / latency)
- `eval/results/false_positives.json` — false-positive rate at the fixed 0.5 threshold
- `eval/results/operating_points.{json,md}` — threshold-agnostic view: FPR at a fixed detection rate, EER, and the 0.2/0.45/0.5/0.55/0.8 sweep
- `eval/results/det_points.json` (+ optional `*.svg`) — operating / threshold curves
- `eval/results/scores/` — raw per-prompt scores, so every number above is reproducible offline

Full harness docs: [`eval/README.md`](https://github.com/bastion-soft/bastion-prompt-protection/blob/main/eval/README.md).